In [1]:
from enviroment import TetrisEnv
import numpy as np
import torch

In [3]:
N_ACCIONES = 40

class Agent:
    def __init__(self):
        pass

    def act(self, state):
        return np.random.choice(N_ACCIONES)

training_loop

In [4]:

# Espacios de acción: 0=rotar_izq, 1=rotar_der, 2=izq, 3=der, 4=bajar, 5=hard_drop

# El resultado de la red neuronal sera una tupla (0-9, 0-3)
def action_move_sequence(action : tuple((int, int))) -> tuple:
    sequence = []

    column = action[0] - 5
    rotation = action[1] - 1

    if rotation < 0:
        for i in range(-rotation):
            sequence.append(0)
    else:
        for i in range(rotation):
            sequence.append(1)

    if column < 0:
        for i in range(-column):
            sequence.append(2)
    else:
        for i in range(column):
            sequence.append(3)

    sequence.append(5)

    return sequence

In [5]:
from torch import nn

N_ACCIONES = 40

class Agent(nn.Module):
    def __init__(self, layers_sizes = [45, 128, 128, 64], dropout_value = 0):

        super().__init__()
        self.dropout_value = dropout_value
        layers = []

        for i in range(len(layers_sizes) - 1):
                
            current_size = layers_sizes[i]
            next_size = layers_sizes[i + 1]

            layers.extend([
                torch.nn.Linear(current_size, next_size),
                torch.nn.LayerNorm(next_size),
                torch.nn.ReLU(),
                torch.nn.Dropout(self.dropout_value)
            ])
        
        self.encoder = torch.nn.Sequential(*layers)
        self.head_q = torch.nn.Linear(layers_sizes[-1], 40)


    def forward(self, state: list) -> torch.Tensor:


        state_tensor = torch.tensor(state, dtype=torch.float32)

        encoded_state = self.encoder(state_tensor)
        q_values = self.head_q(encoded_state)

        return q_values
    
    def act(self, state):

        q_values = self.forward(state)

        action_index = torch.argmax(q_values).item()

        column = action_index // 4
        rotation = action_index % 4

        action = action_move_sequence((column, rotation))

        return action

state engeneering

- heights
- holes
- bumpiness
- wells
- one-hot pieza actual
- one-hot pieza siguiente

In [6]:
def _get_heights(board):
    n_rows, n_cols = board.shape
    heights = np.zeros(n_cols, dtype=np.int32)
    for col in range(n_cols):
        occupied = np.where(board[:, col] == 1)[0]
        if len(occupied) > 0:
            heights[col] = n_rows - occupied[0]
    return heights

def _get_holes(board, heights):
    n_rows, n_cols = board.shape
    holes = np.zeros(n_cols, dtype=np.int32)
    for col in range(n_cols):
        if heights[col] > 0:
            top_row = n_rows - heights[col]
            column_data = board[top_row:, col]
            holes[col] = np.sum(column_data == 0)
    return holes

def _get_bumpiness(heights):
    n_cols = len(heights)
    bumpiness = 0
    if n_cols > 1:
        bumpiness = np.sum(np.abs(np.diff(heights)))
    return bumpiness

def _get_wells(heights):
    n_cols = len(heights)
    wells = np.zeros(n_cols, dtype=np.int32)
    for col in range(n_cols):
        left_height = heights[col - 1] if col > 0 else heights[col]
        right_height = heights[col + 1] if col < n_cols - 1 else heights[col]
        
        min_neighbor = min(left_height, right_height)
        if heights[col] < min_neighbor:
            wells[col] = min_neighbor - heights[col]
    return wells

def _get_onehot_pieces(state):
    current_piece_onehot = np.zeros(7, dtype=np.float32)
    current_piece_onehot[state['current_piece']] = 1
    
    next_piece_onehot = np.zeros(7, dtype=np.float32)
    next_piece_onehot[state['next_piece']] = 1
    return current_piece_onehot, next_piece_onehot

def extract_tetris_features(state):
    """
    Extrae características del estado de Tetris para aprendizaje por refuerzo.
    """
    board = state['board']
    
    heights = _get_heights(board)
    holes = _get_holes(board, heights)
    bumpiness = _get_bumpiness(heights)
    wells = _get_wells(heights)
    current_piece_onehot, next_piece_onehot = _get_onehot_pieces(state)

    # NORMALIZACIÓN
    # Dividimos las variables métricas por el alto del tablero para escalarlas 
    # a un rango aproximado de [0, 1], igualando la escala a los one-hot vectors.
    max_h = float(board.shape[0]) 

    feature_vector = np.concatenate([
        heights / max_h,
        holes / max_h,
        wells / max_h,
        current_piece_onehot,
        next_piece_onehot,
        [bumpiness / max_h]
    ])

    return feature_vector

In [7]:
from torch.cuda import get_device_name, is_available

print("CUDA disponible:", is_available())
if is_available():
    print("Dispositivo CUDA:", get_device_name(0))

device = torch.device("cuda" if is_available() else "cpu")

CUDA disponible: True
Dispositivo CUDA: NVIDIA GeForce RTX 2060


In [8]:
from torch import nn
import random

N_ACCIONES = 40

class DQN_Agent(nn.Module):
    # Ampliamos el tamaño de las capas ocultas para Tetris (de 128 a 256) y añadimos una capa extra
    def __init__(self, layers_sizes = [45, 256, 256, 256, 128], dropout_value = 0.0):
        super().__init__()
        self.dropout_value = dropout_value
        layers = []

        for i in range(len(layers_sizes) - 1):
            current_size = layers_sizes[i]
            next_size = layers_sizes[i + 1]

            layers.extend([
                torch.nn.Linear(current_size, next_size),
                # El uso de LayerNorm aquí es excelente, lo mantenemos
                torch.nn.LayerNorm(next_size),
                torch.nn.ReLU(),
            ])
            # Desactivamos explícitamente el Dropout por defecto, DQN necesita determinismo
            if self.dropout_value > 0:
                layers.append(torch.nn.Dropout(self.dropout_value))
        
        self.encoder = torch.nn.Sequential(*layers)
        # La última capa conecta la salida de la arquitectura con las 40 acciones posibles
        self.head_q = torch.nn.Linear(layers_sizes[-1], N_ACCIONES)

    def forward(self, state: list) -> torch.Tensor:
        state_tensor = torch.tensor(state, dtype=torch.float32).to(device)
        encoded_state = self.encoder(state_tensor)
        q_values = self.head_q(encoded_state)
        return q_values
    
    def act(self, state, epsilon=0.0):
        # Implementación de Epsilon-Greedy para balancear Exploración vs Explotación
        if random.random() < epsilon:
            action_index = random.randint(0, N_ACCIONES - 1)
        else:
            with torch.no_grad():
                q_values = self.forward(state)
                action_index = torch.argmax(q_values).item()

        column = action_index // 4
        rotation = action_index % 4

        action = action_move_sequence((column, rotation))

        return action_index, action

In [9]:
agent = DQN_Agent().to(device)

training loop

In [10]:
gamma = 0.9

In [11]:
import torch.optim as optim
import torch.nn.functional as F
import torch

# Optimizador con un Learning Rate más bajo para estabilidad en DQN
optimizer = optim.Adam(agent.parameters(), lr=1e-4)

# Huber Loss (SmoothL1Loss) es mucho más estable que MSE para DQN frente a outliers
loss_fn = nn.SmoothL1Loss()

def train_dqn(agent, target_agent, optimizer, loss_fn, batch, gamma):

    states = torch.tensor(np.array([exp[0] for exp in batch]), dtype=torch.float32).to(device)
    actions = torch.tensor([exp[1] for exp in batch], dtype=torch.int64).unsqueeze(1).to(device)
    rewards = torch.tensor([exp[2] for exp in batch], dtype=torch.float32).to(device)
    next_states = torch.tensor(np.array([exp[3] for exp in batch]), dtype=torch.float32).to(device)
    # Extraemos el flag 'terminated' para anular la recompensa futura en estados finales
    dones = torch.tensor([exp[4] for exp in batch], dtype=torch.float32).to(device) 
    
    # 1. Valores Q actuales según la Red Principal (Policy Network)
    q_values = agent(states)
    current_q = q_values.gather(1, actions).squeeze(1)
    
    # 2. Valores Q objetivo evaluados usando Double DQN (DDQN)
    with torch.no_grad():
        # A) La Red Principal (agent) decide cuál es la mejor acción futura
        best_actions = agent(next_states).argmax(dim=1, keepdim=True)
        
        # B) La Red Objetivo (target_agent) evalúa el valor de esa acción elegida
        next_q_values = target_agent(next_states)
        max_next_q = next_q_values.gather(1, best_actions).squeeze(1)
        
        # Ecuación de Bellman corrigiendo terminal states (1 - dones)
        target_q = rewards + gamma * max_next_q * (1 - dones)
    
    # 3. Calcular la pérdida
    loss = loss_fn(current_q, target_q)
    
    # 4. Actualizar pesos
    optimizer.zero_grad()
    loss.backward()
    
    # Recorte de gradientes para estabilidad
    torch.nn.utils.clip_grad_norm_(agent.parameters(), max_norm=1.0)
    
    optimizer.step()
    
    return loss.item()

In [12]:
import random
from time import sleep
from collections import deque
import numpy as np
from IPython.display import clear_output
import copy

# PARAMETROS CRITICOS OPTIMIZADOS
N_EXPERIENCE_REPLAY = 50000   
batch_train_size = 128        
train_frequency = 4           
gamma = 0.99                  
TARGET_UPDATE_FREQ = 1000     

# PARÁMETROS EPSILON-GREEDY
epsilon_start = 1.0
epsilon_end = 0.05
# AUMENTADO: 100k era muy poco para Tetris. Le damos 500k pasos para que baje muy poco a poco.
epsilon_decay_steps = 500000  

experience_replay = deque(maxlen=N_EXPERIENCE_REPLAY)
steps_counter = 0

# Variables para tracking/log
log_frequency = 100
episode_rewards_log = []
loss_log = []

# Target Network
target_agent = DQN_Agent().to(device)
target_agent.load_state_dict(agent.state_dict())
target_agent.eval()

for episode in range(10000000):

    env = TetrisEnv(n=20, m=10)

    state, info = env.reset()
    current_feature_vector = extract_tetris_features(state)

    terminated = False
    truncated = False
    current_episode_reward = 0
    current_loss = 0

    while not (terminated or truncated):
        
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * (steps_counter / epsilon_decay_steps))
        
        action_index, action_sequence = agent.act(current_feature_vector, epsilon=epsilon)
        steps_counter += 1

        lines_cleared = 0
        for a in action_sequence:
            state, _, terminated, truncated, info = env.step(a)
            lines_cleared += info.get('step_lines_cleared', 0)
            if terminated or truncated:
                break
                
        next_feature_vector = extract_tetris_features(state)

        # -------------------------------------------------------------
        # NUEVO CALCULO DE RECOMPENSA (DELTAS / DIFERENCIAL)
        # -------------------------------------------------------------
        # ERROR CORREGIDO: Al normalizar el vector entre 0 y 1, estábamos rompiendo
        # nuestros coeficientes de recompensa (-4, -0.5). Hay que multiplicarlos
        # por 20 (altura del tablero) para devolverlos a escala real.
        
        new_height = np.sum(next_feature_vector[0:10]) * 20.0
        new_holes = np.sum(next_feature_vector[10:20]) * 20.0
        new_bumpiness = next_feature_vector[44] * 20.0
        
        old_height = np.sum(current_feature_vector[0:10]) * 20.0
        old_holes = np.sum(current_feature_vector[10:20]) * 20.0
        old_bumpiness = current_feature_vector[44] * 20.0
        
        delta_holes = new_holes - old_holes
        delta_height = new_height - old_height
        delta_bumpiness = new_bumpiness - old_bumpiness
        
        alive_reward = 1 
        game_over_penalty = -50 if terminated else 0
        
        custom_reward = (
            alive_reward + 
            10 * lines_cleared +
            -4 * delta_holes +         
            -0.5 * delta_height +      
            -0.3 * delta_bumpiness +   
            game_over_penalty
        )
        # -------------------------------------------------------------
        
        current_episode_reward += custom_reward

        experience_replay.append((current_feature_vector, action_index, custom_reward, next_feature_vector, terminated))

        if len(experience_replay) >= batch_train_size and steps_counter % train_frequency == 0:
            batch_indices = random.sample(range(len(experience_replay)), batch_train_size)
            batch = [experience_replay[i] for i in batch_indices]
            
            loss = train_dqn(agent, target_agent, optimizer, loss_fn, batch, gamma)  
            current_loss = loss      

        if steps_counter % TARGET_UPDATE_FREQ == 0:
            target_agent.load_state_dict(agent.state_dict())

        current_feature_vector = next_feature_vector

    episode_rewards_log.append(current_episode_reward)
    loss_log.append(current_loss)
    
    if (episode + 1) % log_frequency == 0:
        avg_reward = np.mean(episode_rewards_log[-log_frequency:])
        
        clear_output(wait=True)
        print(f"Episode {episode + 1} | Steps Totales: {steps_counter}")
        print(f"Epsilon actual: {epsilon:.3f}")
        print(f"Average Reward (last {log_frequency}): {avg_reward:.2f}")
        print(f"Last Game Score: {info.get('score', 0)} | Lines: {info.get('lines_cleared', 0)}")
        
env.close()

Episode 318900 | Steps Totales: 17427059
Epsilon actual: 0.050
Average Reward (last 100): 76.71
Last Game Score: 2500 | Lines: 25


KeyboardInterrupt: 

Guardamos los modelos

In [14]:
# torch.save(agent.state_dict(), "dqn_tetris_agent.pth")
# torch.save(target_agent.state_dict(), "dqn_tetris_target_agent.pth")

In [ ]:
agent.load_state_dict(torch.load("dqn_tetris_agent.pth", map_location=device))
target_agent.load_state_dict(torch.load("dqn_tetris_target_agent.pth", map_location=device))

In [ ]:
print("Entrenamiento finalizado.")

In [16]:
from time import sleep

env = TetrisEnv(n=20, m=10, render_mode='human')
state, info = env.reset()

terminated = False

for _ in range(10):  

    while not terminated:
        
        action_index, action_sequence = agent.act(extract_tetris_features(state), epsilon=0.0) 
        for a in action_sequence:
            state, _, terminated, truncated, info = env.step(a)
            sleep(0.1)  # Pequeña pausa para visualizar mejor
            env.render()
            if terminated or truncated:
                break

    

env.close()